# Lab 7 starter: did the pilot change how often customers ordered?

This notebook is standalone. The recap cell rebuilds the pilot window from
Chapter 7.

The lab prompt is at the end of Chapter 7, under **Exercises**.

**Build lab.** The chapter tested whether free shipping changed how much customers
spent per order. This lab tests whether it changed how *often* they ordered. Count
each pilot-eligible customer's orders in the pilot window, including a zero for
those who placed none, then compare the two groups' mean orders per customer with a
permutation test and a Welch t-test, and report the difference with a bootstrap
confidence interval.

**Evaluate lab.** Rerun the analysis excluding each group's single largest
customer. State in two sentences whether the conclusion is unchanged, and why this
check belongs in any analysis whose results will be shared.

## Running this notebook

Run the setup cell below first, then the recap cell, then work down. In Google
Colab nothing needs to be installed beyond that cell. Locally, use the
`pyba-core` environment from Appendix A.

You will enter your work in the cells marked `# TODO`. Everything else is
provided.

**If you are in Google Colab:** this notebook opened read-only from GitHub. Click
**Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before you
edit anything, then work in that copy. It saves to your Google Drive under
*Colab Notebooks*. Edits made to the read-only original are lost when the tab
closes.

In [ ]:
# Setup. Run this cell once per session. It installs this lab's packages;
# on Google Colab it also fetches the course data.
%pip install -q pandas plotly scipy
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Recap: the Chapter 7 pilot window

`window` holds the online orders placed inside the pilot window, March 2 to May 31,
2026, each labelled with its customer's pilot group. This assembly is provided
because the chapter works through it in detail and the lab is about what comes
after it.

Note what `window` counts: one row per **order**. The lab asks about one row per
**customer**, which is the point of Step 1.

In [ ]:
# --- Chapter 7 recap: run this cell first ---
import numpy as np
import pandas as pd
from scipy import stats

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")

totals = orders.groupby("order_id").agg(
    total=("line_total", "sum"),
    date=("order_date", "first"),
    channel=("channel", "first"),
    customer_id=("customer_id", "first"),
)

window = totals[
    (totals["channel"] == "online")
    & (totals["date"] >= "2026-03-02")
    & (totals["date"] <= "2026-05-31")
].merge(snap[["customer_id", "pilot"]], on="customer_id").dropna(subset=["pilot"])

rng = np.random.default_rng(0)
window.groupby("pilot")["total"].agg(["mean", "median", "count"]).round(2)

## Step 1: one row per customer, zeros included

This is the step that decides whether the rest of the lab is correct.

Counting orders inside `window` gives you the customers who ordered at least once.
A customer who was in the pilot and ordered nothing never appears there, and
dropping those customers would bias the comparison: the pilot could have moved
people from one order to zero, and an analysis built only on people who ordered
would never see it.

So start from the customers, not the orders. Take everyone in `snap` with a
non-missing `pilot` value, which is the eligible population. Then attach each
customer's order count from `window`, and fill the missing counts with zero.

`snap["pilot"].notna()` selects the eligible customers. A left merge keeps every
one of them, and `.fillna(0)` turns the absent counts into zeros.

Name the result's two columns `pilot` and `n_orders`, keeping `customer_id`
alongside them. The cell below and every step after it refer to those names.

In [ ]:
# TODO: build one row per eligible customer, with pilot group and order count
#       (customers who placed no orders in the window must appear with 0)
per_customer = None

per_customer.groupby("pilot")["n_orders"].agg(["mean", "count"]).round(3)

**Check your work before continuing.** `per_customer` should have one row for every
eligible customer in `snap`, which is more rows than the number of customers who
actually ordered in the window. If instead it matches the number who ordered, the
zeros went missing and the merge dropped the customers you most need to keep.

In [ ]:
# TODO: confirm that per_customer has one row per eligible customer.
# The placeholder below is not defined, so an unedited cell raises a NameError
# rather than passing silently.
assert CHECK_ROW_COUNT

print(f"{(per_customer['n_orders'] == 0).sum()} eligible customers placed no orders")

## Step 2: the observed difference

Compute the difference in mean orders per customer between the pilot group
(`pilot == 1`) and the control group (`pilot == 0`). One number.

In [ ]:
# TODO: observed difference in mean orders per customer
observed = None
print(f"observed difference {observed:.4f} orders per customer")

## Step 3: the permutation test

The chapter's permutation test shuffles the group labels many times and asks how
often a difference as large as the observed one appears by chance. Its structure:

```python
values = per_customer["n_orders"].values
is_pilot = (per_customer["pilot"] == 1).values

shuffled = []
for _ in range(10_000):
    labels = rng.permutation(is_pilot)
    shuffled.append(values[labels].mean() - values[~labels].mean())
shuffled = np.array(shuffled)
p_value = (np.abs(shuffled) >= abs(observed)).mean()
```

The logic carries over unchanged from the chapter. What differs is the variable
being compared: orders per customer rather than dollars per order. Adapt it, and
print the p-value.

In [ ]:
# TODO: permutation test on mean orders per customer

## Step 4: the Welch t-test

`stats.ttest_ind` with `equal_var=False`, on the two groups' order counts. Two
lines. Print the t-statistic and the p-value.

The two groups differ in size and very likely in spread, which is why the chapter
uses Welch rather than the equal-variance form.

In [ ]:
# TODO: Welch t-test on the two groups

## Step 5: the bootstrap confidence interval

Resample each group with replacement, take the difference of the resampled means,
repeat, and read off the 2.5th and 97.5th percentiles. The chapter's version:

```python
boot = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean()
        for _ in range(10_000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
```

where `a` and `b` are the two groups' values. Report the difference with its
interval.

In [ ]:
# TODO: bootstrap 95% confidence interval for the difference

**What do the three results say together?** Two or three sentences. Note whether
the interval includes zero, and what that implies about the p-values you computed.

---

...

---

## Step 6 (Evaluate lab): drop each group's largest customer

Rerun the comparison with the single highest-count customer removed from each
group, so two customers in total.

The chapter's school-district example is the reason for this check. One unusually
large customer can carry a difference on its own, and a conclusion that depends on
one row is not a conclusion about the pilot.

Recompute the observed difference and at least one of the three tests on the
trimmed data.

In [ ]:
# TODO: drop the highest-count customer from each group, then recompute

**Is the conclusion unchanged, and why does this check belong in any shared
analysis?** Two sentences.

---

...

---

## Before you submit

Run **Restart and run all** and confirm the notebook executes top to bottom with no
errors. Then download as `.ipynb` and upload it to the Lab 7 dropbox.

If you used an AI assistant, add a one-line note naming the tool and what it
generated, per the course policy in Appendix D.